# 🎵 ACE-Step — Kaggle (GPU P100) + full-param веб-интерфейс

Движок ACE-Step + наш **full-param Flask-интерфейс** (`webui/app.py`, ~46 параметров) — всё на Kaggle.
На ПК ставить ничего не нужно: интерфейс открывается в браузере по ссылке `*.trycloudflare.com`.

**Settings:** Accelerator = **GPU P100**, Internet = **On**. У Kaggle ~29 ГБ RAM → **XL (4B)** грузится без OOM.


## 0. Проверка GPU и RAM


In [ ]:
!nvidia-smi
!free -h


## 1. Системные пакеты + cloudflared (Node больше не нужен — UI на Python)


In [ ]:
!apt-get install -y ffmpeg > /dev/null 2>&1
# cloudflared ставим прямым бинарником (dpkg на Kaggle часто падает из-за зависимостей)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version


## 2. Клонирование репо + установка движка и зависимостей

Ставим движок через `pip -e . --no-deps` (используем предустановленный на Kaggle torch),
локальный `nano-vllm` — тоже из репо, плюс FastAPI/uvicorn (REST API движка) и Flask/requests (наш UI).


In [ ]:
%cd /kaggle/working
![ -d ace-step-ui_CPU ] || git clone -q https://github.com/Landers125/ace-step-ui_CPU.git
![ -d ACE-Step-1.5 ] || git clone -q https://github.com/ace-step/ACE-Step-1.5.git
%cd /kaggle/working/ACE-Step-1.5
!pip install -q -e . --no-deps
!pip install -q -e acestep/third_parts/nano-vllm --no-deps
!pip install -q "transformers>=4.51.0,<4.58.0" "diffusers>=0.37.0" "accelerate>=1.12.0" "soundfile>=0.13.1" loguru einops scipy "vector-quantize-pytorch>=1.27.15" diskcache numba pytorch-wavelets pywavelets toml modelscope matplotlib librosa soxr python-dotenv
!pip install -q fastapi "uvicorn[standard]" flask requests
import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())


> Если `CUDA: False` — выполните один раз: `!pip install -q --force-reinstall torch torchaudio --index-url https://download.pytorch.org/whl/cu121`


## 3. Выбор и загрузка модели (веса XL → /kaggle/temp через симлинк)

`acestep-v15-turbo` (2B, быстрая) · `acestep-v15-xl-turbo` (XL 4B, 8 шагов) · `acestep-v15-xl-sft` (XL 4B, 50 шагов).
⚠️ `/kaggle/working` ограничена ~20 ГБ, поэтому веса XL (~20 ГБ) качаем в `/kaggle/temp` (scratch) и подключаем симлинком.


In [ ]:
import os, shutil, subprocess, sys
DIT_MODEL = 'acestep-v15-xl-turbo'   # или 'acestep-v15-turbo' (2B)

shutil.rmtree('/kaggle/working/ACE-Step-1.5/checkpoints', ignore_errors=True)
SCRATCH = '/kaggle/temp/checkpoints'; os.makedirs(SCRATCH, exist_ok=True)
LINK = '/kaggle/working/ACE-Step-1.5/checkpoints'
if os.path.islink(LINK):
    os.remove(LINK)
elif os.path.exists(LINK):
    shutil.rmtree(LINK)
os.symlink(SCRATCH, LINK)
print('checkpoints ->', os.path.realpath(LINK))

_t,_u,_f = shutil.disk_usage('/kaggle/temp'); print('scratch free: %.1f GB' % (_f/1e9))

XL_REPOS = {
  'acestep-v15-xl-turbo': 'ACE-Step/acestep-v15-xl-turbo',
  'acestep-v15-xl-base':  'ACE-Step/acestep-v15-xl-base',
  'acestep-v15-xl-sft':   'ACE-Step/acestep-v15-xl-sft',
}
if DIT_MODEL in XL_REPOS:
    subprocess.run([sys.executable,'-m','pip','install','-q','-U','huggingface_hub[hf_xet]','hf_xet'], check=False)
    from huggingface_hub import snapshot_download
    dest = os.path.join(LINK, DIT_MODEL)
    print('Скачиваю', XL_REPOS[DIT_MODEL], '->', os.path.realpath(dest))
    snapshot_download(repo_id=XL_REPOS[DIT_MODEL], local_dir=dest, max_workers=4)
    subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub>=0.34.0,<1.0'], check=False)
    print('Готово, файлов:', len(os.listdir(dest)))
else:
    print('2B-модель скачается автоматически при первой генерации.')


## 4. Запуск REST API движка (порт 8001)

`ACESTEP_NO_INIT=true` — ленивая загрузка: модель грузится при первом запросе (именно выбранная в UI),
поэтому не тратим VRAM на дефолтную 2B перед XL. LLM по умолчанию выключен.


In [ ]:
import subprocess, os, time
ENGINE = '/kaggle/working/ACE-Step-1.5'
e = os.environ.copy()
e['ACESTEP_API_HOST'] = '0.0.0.0'
e['ACESTEP_API_PORT'] = '8001'
e['ACESTEP_NO_INIT'] = 'true'
subprocess.Popen('acestep-api > /kaggle/working/engine.log 2>&1', shell=True, cwd=ENGINE, env=e)
print('REST API движка стартует на :8001, ждём ~40 сек...'); time.sleep(40)
!tail -n 25 /kaggle/working/engine.log


> Ждём строку `Uvicorn running on http://0.0.0.0:8001`. Если ещё грузится — повторите `!tail -n 25 /kaggle/working/engine.log`.


## 5. Запуск нашего full-param интерфейса (порт 5000)


In [ ]:
import subprocess, os, time
UI = '/kaggle/working/ace-step-ui_CPU/webui'
e = os.environ.copy()
e['ACE_BASE_URL'] = 'http://localhost:8001'
e['PORT'] = '5000'
subprocess.Popen('python app.py > /kaggle/working/webui.log 2>&1', shell=True, cwd=UI, env=e)
print('Веб-интерфейс стартует на :5000, ждём 8 сек...'); time.sleep(8)
!tail -n 15 /kaggle/working/webui.log


## 6. Публичный туннель (порт 5000)


In [ ]:
import subprocess, time, re
subprocess.Popen('cloudflared tunnel --url http://localhost:5000 --no-autoupdate > /kaggle/working/cf.log 2>&1', shell=True)
url=None
for _ in range(40):
    time.sleep(2)
    try: log=open('/kaggle/working/cf.log').read()
    except Exception: log=''
    m=re.search('https://[a-z0-9-]+[.]trycloudflare[.]com', log)
    if m: url=m.group(0); break
print('ОТКРОЙ В БРАУЗЕРЕ:', url or 'см. /kaggle/working/cf.log')


## ✅ Готово

1. Открой ссылку `*.trycloudflare.com` из ячейки 6 в браузере.
2. В интерфейсе выбери модель `acestep-v15-xl-turbo` (или 2B-turbo).
3. На P100 для XL: **Batch Size = 1**, длительность 30–120 сек.
4. Логи: `!tail -n 60 /kaggle/working/engine.log` и `webui.log`.

⚠️ Веса XL в `/kaggle/temp` очищаются при завершении сессии — либо качать заново, либо сохранить в приватный Kaggle Dataset.
